# Reproducing Ling et al. (2023/2024) from dandiset 001170

**Paper.** Ling D, Moss EH, Smith CL, Deng F, Kroeger R, Reimer JL, Raman B, Arenkiel BR.
Conserved neural dynamics and computations across species in olfaction. bioRxiv
[doi:10.1101/2023.04.24.538157](https://doi.org/10.1101/2023.04.24.538157). This attempt used
the full text of version 2 of the preprint (2024-02-23), which is open access (CC BY-ND); its
Table S1 and key resources table are images that could not be fetched. The paper was later
published in iScience (2026, doi:10.1016/j.isci.2026.115373) with a new title; the claims tested
here are those of the preprint, which the dandiset names.

**Dandiset.** [DANDI:001170](https://dandiarchive.org/dandiset/001170/0.260528.1016), version
0.260528.1016. The paper cites no DANDI version (its data statement promises Figshare). This is the
most recent published version; the only other one, 0.251111.1532, holds the same 12 assets (same
asset ids, sizes and digests, shown in the data check) and differs only in metadata (anatomy terms,
keywords, funders, affiliations and a longer description), none of which the analysis uses.

**What the dandiset holds.** Two-photon imaging of the dorsal olfactory bulb of awake mice: raw
fluorescence of glomerular ROIs, the raw movies, a trials table of odor valve openings, and
treadmill (and in one file respiration) traces. Seven sessions from four mice follow the paper's
block protocol (Figures 2, 4, 6, 7); five sessions from five mice (one of them, sub-134, also in the
block sessions) follow a random-order protocol with 11 odors. The locust recordings of the paper are not in the
dandiset; they are on Figshare with the paper's code.

**Outside inputs.** The paper's Figshare record
([doi:10.6084/m9.figshare.31435543.v2](https://doi.org/10.6084/m9.figshare.31435543.v2)): its
MATLAB scripts (read for method choices), the deposited mouse array `data1_mouse_glom198.mat`
(used only by the link claim), and the locust files `data2_locust_PN66.mat`,
`data4_locust_PN85.mat` and `data5_locust_PN85.mat` (used by the locust claims, data elsewhere).
Each file is checked against the md5 the record gives.

Random numbers (random vectors and label permutations) come from a generator seeded per claim.

| Claim | Data | Status |
|---|---|---|
| mouse-sessions | dandiset | tested |
| figshare-trace-link | with inputs | tested |
| fig2h-onoff-angle-mouse | dandiset | tested |
| fig2f-onoff-anticorrelated-mouse | dandiset | tested |
| fig4b-reference-odor-mouse | dandiset | tested |
| fig4d-clustering-mouse | dandiset | tested |
| fig6e-contrast-msc | dandiset | tested |
| fig6e-contrast-ace | dandiset | blocked: no sequence ends in acetophenone |
| fig7d-persistent-anticorrelation | dandiset | tested |
| fig7h-iti-knn-mouse | dandiset | tested |
| fig7j-iti-timebin-mouse | dandiset | tested |
| figs7e-ibi-chance | dandiset | tested |
| figs8e-random-order-decline | dandiset | tested |
| figs2-pid-decay | dandiset | blocked: no PID recordings |
| fig1g-onoff-angle-locust | elsewhere (Figshare) | tested |
| figs4a-reference-odor-locust | elsewhere (Figshare) | tested |
| fig5e-contrast-locust | elsewhere (Figshare) | tested |
| fig7f-iti-knn-locust | elsewhere (Figshare) | tested |
| fig7i-iti-timebin-locust | elsewhere (Figshare) | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

import olfdyn

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
urls = olfdyn.asset_urls()
with ThreadPoolExecutor(3) as pool:
    sessions = list(pool.map(lambda p: olfdyn.reduce_session(p, urls[p]), sorted(urls)))
for s in sessions:
    s["name"] = f"sub-{s['subject']} ses-{s['session']}"
block = [s for s in sessions if s["protocol"] == "block"]
random_sessions = [s for s in sessions if s["protocol"] == "random"]
print(f"loaded {len(sessions)} files from dandiset {olfdyn.DANDISET_ID} version {olfdyn.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
from dandi.dandiapi import DandiAPIClient
with DandiAPIClient() as client:
    versions = [v.identifier for v in client.get_dandiset(olfdyn.DANDISET_ID).get_versions()]
print("Versions on the archive:", versions)
tables = {v: olfdyn.asset_table(v) for v in versions if v != "draft"}
first, last = tables["0.251111.1532"], tables[olfdyn.DANDISET_VERSION]
print("Assets per version:", {v: len(t) for v, t in tables.items()})
print("0.251111.1532 and", olfdyn.DANDISET_VERSION, "hold the same paths, asset ids, sizes and digests:",
      first.equals(last))
print(f"Total size: {last['size'].sum() / 1e9:.1f} GB")

In [ ]:
info = pd.DataFrame([{
    "session": s["name"], "sex": s["sex"], "dob": s["dob"], "date": s["start"][:10], "protocol": s["protocol"],
    "rois": s["n_rois"], "rois_per_fov": s["rois_per_fov"], "frames": s["n_frames"],
    "frame_hz": round(s["frame_hz"], 2), "unit": s["unit"], "indicator": s["indicator"],
    "location": s["location"], "same_t_3fov": s["same_timestamps"], "behavior": ",".join(s["behavior"]),
    "description": s["subject_description"]} for s in sessions])
print(info.to_string(index=False))
print("\nSessions per protocol:", info.protocol.value_counts().to_dict())
print("Mice per protocol:", info.assign(m=info.session.str.split().str[0]).groupby("protocol").m.nunique().to_dict())
print("Frame rates (Hz): min", info.frame_hz.min(), "max", info.frame_hz.max(), "(Methods: 15-18 Hz; code: 16)")
print("Session start times:", sorted({s['start'][11:] for s in sessions}))

Twelve sessions from eight mice. The fluorescence is raw F (unit "n.a."), not the dF/F the Methods
describe, and no description says how it was extracted. The indicator (GCaMP6f) and the 920 nm
excitation agree with the Methods; the genotype (Thy1-GCaMP6f) and the ages (two to four months)
are not stated, though dates of birth are. Every session starts at 00:00, so the time of day is
missing. The three fields of view of a session share one timestamp array. Frame rates are 13.7 to
17.8 Hz, against the Methods' 15 to 18 Hz and the code's nominal 16.

In [ ]:
rows = []
for s in sessions:
    tp = s["two_photon"]
    fl_dt = 1 / s["frame_hz"]
    for k, (n, dt) in tp.items():
        rows.append({"session": s["name"], "series": k.replace("two_photon_series_", ""), "movie_frames": n,
                     "trace_frames": s["n_frames"], "ratio": round(s["n_frames"] / n, 3),
                     "movie_dt_ms": round(1000 * dt, 1), "trace_dt_ms": round(1000 * fl_dt, 1)})
tp = pd.DataFrame(rows)
print(tp[tp.series.str.startswith("FOV1")].to_string(index=False))
print("\nSessions whose raw movie holds fewer frames than the traces:",
      tp[tp.ratio != 1].session.unique().tolist())

In three sessions (sub-134 ses-22, sub-141 ses-5, sub-141 ses-8) the raw TwoPhotonSeries holds half
the frames of the fluorescence traces over the same span, at twice the frame interval. The
conversion notes in the conversion repository give 182000 frames at 17.8 Hz for sub-134 ses-22, the
count of its traces, so the movies of these three files are missing every other frame (or were
written at half rate). The analysis uses the traces only.

In [ ]:
rows = []
for s in sessions:
    tr, p = s["trials"], s["pres"]
    dur = tr.stop_time - tr.start_time
    rows.append({"session": s["name"], "protocol": s["protocol"], "rows": len(tr),
                 "empty_rows": int((tr.odorant == olfdyn.EMPTY).sum()),
                 "row_s": f"{dur.min():.2f}-{dur.max():.2f}", "presentations": len(p),
                 "pulse_rows": sorted(p.rows.unique().tolist()),
                 "odors": len(p.odor.unique()), "concentrations": sorted(tr.concentration.unique().tolist()),
                 "conc_of_empty": sorted(tr.concentration[tr.odorant == olfdyn.EMPTY].unique().tolist()),
                 "first_trial_s": round(tr.start_time.min() - s["clock"]["t_first"], 1),
                 "after_last_s": round(s["clock"]["t_last"] - tr.stop_time.max(), 1)})
print(pd.DataFrame(rows).to_string(index=False))
names = sorted({n for s in sessions for n in s["trials"].odorant.unique()})
print("\nOdorant names in the files:", names)

Each trials-table row is one second of valve opening: a 5-s presentation is five rows, and in some
sessions the inter-trial seconds are rows named "Empty" (sub-141 ses-5 gives these rows a
concentration of 0.1, the other sessions 0). Two block sessions (sub-134 ses-22, sub-141 ses-1) have
no Empty rows. Presentations are rebuilt here by joining consecutive rows of one odorant. The
concentration has no unit. Odorant names are spelled differently across sessions (for example
"Anisole" and "anisole"); "2-methypyrazine" is a misspelling. Names of the random-order odors differ
from the paper's list: the files have 2-ethylphenol and anisaldehyde where the Methods list
2-ethylpyrizine (2ep) and isoamyl acetate (asa).

In [ ]:
for s in block:
    p = s["pres"]
    summary = []
    for b, g in p.groupby("block"):
        kinds = g.kind.unique().tolist()
        odors = "/".join(g[g.kind != "seq_second"].odor.unique())
        second = "/".join(g[g.kind == "seq_second"].odor.unique())
        pulse = (g.offset - g.onset).median()
        isi = np.median(np.diff(g[g.kind != "seq_second"].onset)) if len(g) > 1 else np.nan
        pair = (g[g.kind == "seq_second"].onset.to_numpy() - g[g.kind == "seq_first"].onset.to_numpy()).mean() \
            if "seq_first" in kinds else np.nan
        summary.append(f"{'sol' if kinds == ['solitary'] else 'seq'}:{odors}{('->' + second) if second else ''}"
                       f" n={int((g.kind != 'seq_second').sum())} pulse={pulse:.2f}s isi={isi:.1f}s"
                       + (f" pair_lag={pair:.2f}s" if pair == pair else ""))
    gaps = np.diff([g.onset.min() for _, g in p.groupby("block")])
    ends = [g.onset.max() for _, g in p.groupby("block")]
    ibis = [p[p.block == b + 1].onset.min() - ends[b] for b in range(len(ends) - 1)]
    print(s["name"], "|", "; ".join(summary))
    print("    time from a block's last onset to the next block's first onset (s):", np.round(ibis, 0).tolist())

All seven block sessions hold four solitary blocks (methyl salicylate, acetophenone, allyl sulfide,
eugenol; ten 5-s pulses every 60 s, so 55 s between pulses) and three sequential blocks in which
acetophenone, allyl sulfide or eugenol is followed 6 s after its onset (1 s after its offset) by
methyl salicylate, ten pairs every 60 s. Blocks are about 15 min apart (the last onset of a block
to the next block's first onset is 961 to 967 s, and 913 s between the last sequential and the
first solitary block in sub-148 and sub-149). This agrees with the Methods. No block presents
acetophenone, or any odor other than methyl salicylate, as the second odor of a pair. The order of
blocks differs between mice: sub-134 and sub-141 start with the solitary blocks, sub-148 and
sub-149 with the sequential ones.

In [ ]:
for s in random_sessions:
    p = s["pres"]
    isi = np.diff(p.onset)
    print(f"{s['name']}: {len(p)} presentations of {p.odor.nunique()} odorants incl. Empty, pulse "
          f"{(p.offset - p.onset).median():.2f} s, onset interval median {np.median(isi):.2f} s "
          f"(range {isi.min():.2f}-{isi.max():.2f}); presentations per odor (Empty excluded) "
          f"{p[p.odorant != olfdyn.EMPTY].odor.value_counts().min()}-{p[p.odorant != olfdyn.EMPTY].odor.value_counts().max()}")
    print("    concentrations by odor:",
          {o: sorted(g.concentration.unique().tolist()) for o, g in p[p.odorant != olfdyn.EMPTY].groupby("odor")})
seqs = {s["name"]: tuple(s["pres"].odor.str.lower()) for s in random_sessions}
base = seqs[random_sessions[0]["name"]]
print("\nRandom-order sessions whose first 200 presentations match", random_sessions[0]["name"], ":",
      [n for n, q in seqs.items() if q[:200] == base[:200]])

The five random-order sessions present 1-s pulses every 20 s, as the Methods' random-order
experiment does, and share one fixed odor sequence. In three of them (sub-124, sub-125, sub-126)
each odor has its own concentration value, from 0.01 to 0.9, and two odors (2-ethylphenol and
anisaldehyde) have 0.0, the value given to the blank "Empty"; the Methods give one dilution for all
odors. The other two sessions give 0.01 to every odor. None of the five has the 286 glomeruli the
Methods give for Figure S8D. In sub-124 ses-3 the last trial ends 8.9 s after the last imaging
frame.

In [ ]:
# Recordings deposited twice: frame counts, ROI counts and the first frames of the traces.
keys = pd.DataFrame([{"session": s["name"], "frames": s["n_frames"], "rois": s["n_rois"],
                      "first_frame_mean": float(np.round(s["F_median"].mean(), 3))} for s in sessions])
dup_frames = keys[keys.duplicated("frames", keep=False)]
print("Sessions sharing a frame count:\n", dup_frames.to_string(index=False))
pairs = []
for i, a in enumerate(sessions):
    for b in sessions[i + 1:]:
        if a["n_rois"] == b["n_rois"] or np.isclose(a["F_median"].mean(), b["F_median"].mean()):
            pairs.append((a["name"], b["name"]))
print("Pairs sharing a ROI count or the same mean of median F:", pairs)
same_trials = [(a["name"], b["name"]) for i, a in enumerate(sessions) for b in sessions[i + 1:]
               if len(a["trials"]) == len(b["trials"]) and np.allclose(a["trials"].start_time, b["trials"].start_time)]
print("Pairs with identical trial start times:", same_trials)

Two sessions share a frame count (sub-125 ses-1 and sub-141 ses-5, 147000), but they differ in ROI
count and protocol. No two sessions share ROI counts or trial times, so no recording appears to be
deposited twice. Sessions of one mouse (sub-134, sub-141, sub-148) are analysed separately; nothing
says that their ROIs are the same glomeruli.

In [ ]:
# Clock: the traces' timestamps and the trials table are both on the odor clock (conversion code).
# Block sessions: latency of the population response after each onset, from the trial-averaged
# absolute change of F relative to the 2 s before onset.
rows = []
for s in block:
    x = olfdyn.dff_pre(s["sol"])
    curve = np.abs(x.mean(axis=0)).mean(axis=1)           # frames
    pre = curve[:32]
    thr = pre.mean() + 5 * pre.std()
    after = np.flatnonzero(curve[32:] > thr)
    rows.append({"session": s["name"], "latency_s": (after[0] / 16) if len(after) else np.nan,
                 "peak_s": float(np.argmax(curve[32:112]) / 16), "pre_level": float(pre.mean()),
                 "peak_level": float(curve[32:112].max())})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# Random-order sessions: decode the odor from 0.25 to 2 s after onset, with the true labels and with
# the labels of the presentation before (a contrast between conditions).
rows = []
for s in random_sessions:
    q = s["rnd_pres"]
    x = olfdyn.dff_pre(s["rnd"])[:, 36:64].mean(axis=1)
    nb = olfdyn.knn_neighbours(x, np.arange(len(x)), 5, "cosine")
    true = q.odor.to_numpy()
    prev = q.previous.fillna("none").to_numpy()
    rows.append({"session": s["name"], "n": len(q), "acc_true": (olfdyn.knn_predict(nb, true) == true).mean(),
                 "acc_previous_label": (olfdyn.knn_predict(nb, prev) == prev).mean(), "chance": 1 / 11})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In every block session the population response rises 0.19 to 0.31 s after the valve onset and
peaks within the 5 s pulse (0.5 to 4.75 s). In the random-order sessions the odor is decoded from
the 2 s after onset at 25.5% to 83.6% with the true labels and at 5.5% to 7.3% with the label of
the previous presentation (chance 9.1%). This is consistent with the traces and the trials table sharing a clock;
a constant offset shorter than about half a second cannot be excluded from these outputs.

## mouse-sessions

> The mouse block-protocol data (four odors, 5 s pulses, 55 s between trials, blocks of ten, 15 min
> between blocks) come from seven imaging sessions in four mice, with a median of 215 glomeruli per
> imaging session.

In [ ]:
n_block = len(block)
mice = sorted({s["subject"] for s in block})
rois_block = [s["n_rois"] for s in block]
rois_all = [s["n_rois"] for s in sessions]
print("Block-protocol sessions:", n_block, [s["name"] for s in block])
print("Mice:", len(mice), mice)
print("ROIs per block session:", dict(zip([s["name"] for s in block], rois_block)))
print("(a) median over the block sessions:", np.median(rois_block))
print("(b) median over all 12 sessions:", np.median(rois_all), "range", min(rois_all), "-", max(rois_all))
print("Expected 7 sessions, 4 mice, median 215 +/- 21.5")

**Result: reproduced.** The trials tables give seven block-protocol sessions from four mice (134,
141, 148, 149). Over these sessions the median ROI count is 210, within 215 +/- 21.5 (reading a);
over all twelve sessions it is 296 (reading b). The ROI counts of sub-141 ses-8 (198), sub-149
ses-1 (219) and sub-148 ses-3 (318) equal the glomerulus counts the Methods give for Figures 2F
and 4C, 6D, and 7, so these sessions are taken to be those figures' sessions. This is an
assumption by count: no identifier links the paper's datasets to the dandiset's sessions.

## figshare-trace-link

> The dandiset's raw fluorescence of the 198-glomerulus session is the data the paper deposited
> for its Figure 2D to 2F and 4C (Figshare data1_mouse_glom198.mat), up to the normalisation, with
> glomeruli in the same order.

In [ ]:
dep_mat = olfdyn.figshare_mat("data1_mouse_glom198.mat")["blockData"]
dep_blocks = {b: getattr(dep_mat, f"block{b}") for b in range(1, 8)}
print("Deposited blocks (trials x frames x glomeruli):", {b: v.shape for b, v in dep_blocks.items()})
link = [s for s in sessions if s["n_rois"] == 198]
print("Dandiset sessions with 198 ROIs:", [s["name"] for s in link])
s198 = link[0]
print("Its solitary blocks in recorded order:", s198["sol_pres"].groupby("block").odor.first().tolist())
dep = np.concatenate([dep_blocks[b] for b in range(1, 5)]).astype(np.float64)     # 40 x 912 x 198


def within_trial_r(a, b):
    a = a - a.mean(axis=1, keepdims=True)
    b = b - b.mean(axis=1, keepdims=True)
    return (a * b).sum(axis=1) / np.sqrt((a ** 2).sum(axis=1) * (b ** 2).sum(axis=1))   # trials x glomeruli


r_frames = within_trial_r(s198["sol_frames"].astype(np.float64), dep)
r_seconds = within_trial_r(s198["sol"][:, :912].astype(np.float64), dep)
print(f"\nWithin-trial r, windows counted in frames as the code does: median {np.median(r_frames):.3f} "
      f"(5th-95th percentile {np.percentile(r_frames, 5):.3f} to {np.percentile(r_frames, 95):.3f})")
print("  median by deposited block:", [round(float(np.median(r_frames[i * 10:(i + 1) * 10])), 3) for i in range(4)])
print(f"Within-trial r, windows placed in seconds (variant): median {np.median(r_seconds):.3f}")

# ROI order: for each deposited glomerulus, the dandiset ROI it correlates with best (trial means removed).
a = (s198["sol_frames"] - s198["sol_frames"].mean(axis=1, keepdims=True)).reshape(-1, 198).astype(np.float64)
b = (dep - dep.mean(axis=1, keepdims=True)).reshape(-1, 198)
a, b = (a - a.mean(0)) / a.std(0), (b - b.mean(0)) / b.std(0)
C = b.T @ a / len(a)
best = C.argmax(axis=1)
print(f"Deposited glomeruli whose best-matching dandiset ROI has the same index: {np.mean(best == np.arange(198)):.1%}")

# What the deposited normalisation is close to.
z = olfdyn.zscore_trials(s198["sol_frames"].astype(np.float64))
print(f"Deposited values against F z-scored within each trial window: r = {np.corrcoef(z.ravel(), dep.ravel())[0, 1]:.3f}, "
      f"median absolute difference {np.median(np.abs(z - dep)):.3f} (deposited median absolute value {np.median(np.abs(dep)):.3f})")

**Result: reproduced.** With windows counted in frames as the code does, the median within-trial
r between the dandiset's raw F of sub-141 ses-8 and the deposited array is 0.968 (5th percentile
0.834; 0.947 to 0.994 by block), and every deposited glomerulus correlates best with the dandiset
ROI of the same index. With windows placed in seconds the median r is 0.307: the code counts 16
frames per second in a session imaged at 15.5 Hz, so its windows drift by about 1.8 s over a
912-frame trial. The deposited values are close to F z-scored within each trial window (r = 0.927).

## Responses of the block sessions

Trial-averaged responses to the four solitary odors in each block session, used by the next
claims. Frames are on a 16 Hz grid from 2 s before onset; ON is frames 32 to 112 (0 to 5 s after
onset), OFF is frames 112 to 192 (0 to 5 s after offset).

In [ ]:
PRE, ON, OFF = slice(0, 32), slice(32, 112), slice(112, 192)
resp = {}          # (session, odor) -> trial-averaged dF/F, frames x ROIs (F0 = 2 s before onset)
resp_z = {}        # the same from per-trial z-scored F (variant)
for s in block:
    x = olfdyn.dff_pre(s["sol"])
    z = olfdyn.zscore_trials(s["sol"][:, :912])
    for odor in olfdyn.BLOCK_ODORS:
        sel = (s["sol_pres"].odor == odor).to_numpy()
        resp[s["name"], odor] = x[sel].mean(axis=0).astype(np.float64)
        resp_z[s["name"], odor] = z[sel].mean(axis=0).astype(np.float64)
print("Trials per session and odor:", sorted({int((s['sol_pres'].odor == o).sum()) for s in block for o in olfdyn.BLOCK_ODORS}))
print("dF/F range over all trial-averaged frames:", round(min(v.min() for v in resp.values()), 3), "to",
      round(max(v.max() for v in resp.values()), 3))
print("F0 (2 s pre-onset mean) range over trials and ROIs:",
      round(min(s["sol"][:, PRE].mean(axis=1).min() for s in block), 1), "to",
      round(max(s["sol"][:, PRE].mean(axis=1).max() for s in block), 1))

## fig2h-onoff-angle-mouse

> In the mouse OB the angle between glomerular ON and OFF response vectors is greater than 90
> degrees for each odor (mean 121.8 +/- 2.98 SEM, 4 odors, 7 sessions), so ON and OFF responses are
> inverted relative to each other.

In [ ]:
rng = np.random.default_rng(2)
rows = []
for s in block:
    for odor in olfdyn.BLOCK_ODORS:
        r, rz = resp[s["name"], odor], resp_z[s["name"], odor]
        med = olfdyn.median_subtracted(r)
        main = olfdyn.onoff_angles(med[ON], med[OFF], center=True)
        noise = rng.standard_normal((160, r.shape[1]))
        rows.append({"session": s["name"], "mouse": s["subject"], "odor": odor, "on_off": main["on_off"],
                     "on_on": main["on_on"], "off_off": main["off_off"],
                     "uncentred": olfdyn.onoff_angles(med[ON], med[OFF], center=False)["on_off"],
                     "no_median": olfdyn.onoff_angles(r[ON], r[OFF], center=True)["on_off"],
                     "uncentred_no_median": olfdyn.onoff_angles(r[ON], r[OFF], center=False)["on_off"],
                     "zscored": olfdyn.onoff_angles(olfdyn.median_subtracted(rz)[ON], olfdyn.median_subtracted(rz)[OFF])["on_off"],
                     "random": olfdyn.onoff_angles(noise[:80], noise[80:])["on_off"]})
ang = pd.DataFrame(rows)
print(ang.round(1).to_string(index=False))
cols = ["on_off", "on_on", "off_off", "uncentred", "no_median", "uncentred_no_median", "zscored", "random"]
summary = pd.DataFrame({c: [ang[c].mean(), ang[c].std(ddof=1) / np.sqrt(len(ang)), ang[c].min(), ang[c].max(),
                            int((ang[c] > 90).sum())] for c in cols},
                       index=["mean", "sem_28", "min", "max", "n_above_90"]).T
print("\nOver the 28 session-odor values:\n", summary.round(2).to_string())
print("\nMean ON-OFF angle per odor (over sessions):", ang.groupby("odor").on_off.mean().round(1).to_dict())
print("Mean ON-OFF angle per mouse:", ang.groupby("mouse").on_off.mean().round(1).to_dict())
print("Mean ON-OFF angle per session:", ang.groupby("session").on_off.mean().round(1).to_dict())
print("SEM over the 4 odor means:", round(ang.groupby("odor").on_off.mean().std(ddof=1) / 2, 2))
cos = []
for (name, odor), r in resp.items():
    med = olfdyn.median_subtracted(r)
    x = np.vstack([med[ON], med[OFF]]); x = x - x.mean(axis=0)
    a, b = x[:80].mean(0), x[80:].mean(0)
    cos.append(a @ b / np.linalg.norm(a) / np.linalg.norm(b))
print("Cosine between the mean ON and mean OFF vectors after centring over the 160 frames: min", round(min(cos), 6), "max", round(max(cos), 6))
print("Expected 121.8 +/- 2.98")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for i, c in enumerate(["on_off", "uncentred", "no_median", "zscored", "random"]):
    ax.scatter(np.full(len(ang), i) + np.linspace(-0.2, 0.2, len(ang)), ang[c], s=12)
ax.axhline(90, color="grey", lw=0.8)
ax.axhline(121.8, color="k", lw=0.8, ls="--")
ax.set_xticks(range(5), ["main", "uncentred", "no median", "z-scored", "random"])
ax.set_ylabel("median ON-OFF angle (deg)")
ax.set_ylim(0, 180)
plt.tight_layout()
print("All plotted values lie in 0-180 degrees:", bool(ang[["on_off", "uncentred", "no_median", "zscored", "random"]].stack().between(0, 180).all()))

**Result: partially reproduced.** With the code's procedure the median ON-OFF angle exceeds 90
degrees in all 28 session-odor pairs (101.7 to 153.9) and in each odor's mean (111.0 to 140.0), but
the mean, 127.9 degrees (SEM 3.2 over the 28 values), is outside 121.8 +/- 2.98. The angle above 90
degrees comes with centring the vectors over the 10 s analysed: after centring, the mean ON vector
is exactly opposite the mean OFF vector (cosine -1 in every case), and without centring the mean
angle is 71.8 degrees, with 2 of 28 above 90. Without the median subtraction the mean is 124.2, and
with per-trial z-scored traces 119.5, both within tolerance. Random vectors give 90.4, and the ON-ON
and OFF-OFF medians average 54.0 and 51.9 degrees.

## fig2f-onoff-anticorrelated-mouse

> Time-averaged glomerular ON and OFF responses to the same odor are anticorrelated; glomeruli
> active during the odor tend to be suppressed after it and the reverse.

In [ ]:
rows = []
for s in block:
    for odor in olfdyn.BLOCK_ODORS:
        r = resp[s["name"], odor]
        med = olfdyn.median_subtracted(r)
        rz = resp_z[s["name"], odor]
        rows.append({"session": s["name"], "mouse": s["subject"], "odor": odor,
                     "r_main": olfdyn.corr(med[ON].mean(0), med[OFF].mean(0)),
                     "r_no_median": olfdyn.corr(r[ON].mean(0), r[OFF].mean(0)),
                     "r_zscored": olfdyn.corr(rz[ON].mean(0), rz[OFF].mean(0))})
cr = pd.DataFrame(rows)
print(cr.round(3).to_string(index=False))
for c in ["r_main", "r_no_median", "r_zscored"]:
    w = stats.wilcoxon(cr[c])
    print(f"\n{c}: median {cr[c].median():.3f}, {int((cr[c] < 0).sum())} of {len(cr)} below 0, Wilcoxon p = {w.pvalue:.2g}")
    print("  median per odor:", cr.groupby("odor")[c].median().round(3).to_dict())
    print("  median per mouse:", cr.groupby("mouse")[c].median().round(3).to_dict())

**Result: not reproduced.** The time-averaged ON and OFF vectors are positively correlated in all
28 session-odor pairs (median r = 0.481, Wilcoxon p = 7.5e-09), for every odor (medians 0.391 to
0.535) and every mouse (0.245 to 0.596). Subtracting the median across glomeruli leaves r
unchanged, as it shifts every glomerulus of a vector by the same amount. With per-trial z-scored
traces, the normalisation of the paper's deposited arrays, the median r is 0.068 (12 of 28 below
0, p = 0.33).

## fig4b-reference-odor-mouse

> In the mouse OB the ON response to a reference odor is closer in angle to the ON responses of
> the other odors than to any OFF response, including its own, and this holds for every odor of the
> panel.

In [ ]:
def reference_angles(name, top=3):
    blocks = []
    for odor in olfdyn.BLOCK_ODORS:
        med = olfdyn.median_subtracted(resp[name, odor])
        blocks += [med[ON], med[OFF]]
    x = np.vstack(blocks)
    x = x - x.mean(axis=0)
    if top:
        u, sv, vt = np.linalg.svd(x, full_matrices=False)
        x = x @ vt[:top].T
    means = np.stack([x[i * 80:(i + 1) * 80].mean(0) for i in range(8)])   # odor1 ON, odor1 OFF, ...
    on, off = means[0::2], means[1::2]
    out = []
    for i, odor in enumerate(olfdyn.BLOCK_ODORS):
        a_on = olfdyn.angles(on[i:i + 1], on)[0]
        a_off = olfdyn.angles(on[i:i + 1], off)[0]
        others = np.delete(a_on, i)
        out.append({"session": name, "ref": odor, "on_on_mean": others.mean(), "on_off_mean": a_off.mean(),
                    "on_on_max": others.max(), "on_off_min": a_off.min(), "own_off": a_off[i]})
    return out


for top, label in [(3, "top three PCs (main)"), (None, "full dimension (variant)")]:
    ra = pd.DataFrame([r for s in block for r in reference_angles(s["name"], top)])
    print(f"--- {label}")
    print(ra.round(1).to_string(index=False))
    for odor, g in ra.groupby("ref", sort=False):
        t = stats.ttest_rel(g.on_on_mean, g.on_off_mean)
        print(f"ref {odor}: ON-ON {g.on_on_mean.mean():.1f}, ON-OFF {g.on_off_mean.mean():.1f}, paired t p = {t.pvalue:.2g}; "
              f"sessions where max ON-ON < min ON-OFF: {int((g.on_on_max < g.on_off_min).sum())} of {len(g)}")
    if top == 3:
        ra_main = ra

**Result: partially reproduced.** Reading (a) holds for one reference odor: the acetophenone ON
vector is closer to the other ON vectors (105.7 degrees) than to the OFF vectors (127.5; paired p =
0.018). For methyl salicylate and eugenol the two do not differ (p = 0.76 and 0.83), and the allyl
sulfide ON vector is closer to the OFF vectors (ON-ON 112.1, ON-OFF 59.6, p = 0.00035), the reverse
of the claim. Reading (b) holds in none of the seven sessions for any reference odor. The
full-dimensional variant gives the same pattern (acetophenone p = 0.0096 in the claimed direction,
allyl sulfide p = 0.0013 reversed).

## fig4d-clustering-mouse

> Hierarchical clustering of glomerular responses cleanly separates ON responses from OFF
> responses.

In [ ]:
rows = []
for s in block:
    for label, src in [("dff", resp), ("zscored", resp_z)]:
        vec = np.vstack([np.vstack([src[s["name"], o][ON].mean(0), src[s["name"], o][OFF].mean(0)])
                         for o in olfdyn.BLOCK_ODORS])            # ON, OFF per odor
        cl = fcluster(linkage(pdist(vec, "cosine"), "complete"), 2, "maxclust")
        on_cl, off_cl = set(cl[0::2]), set(cl[1::2])
        rows.append({"session": s["name"], "mouse": s["subject"], "rois": s["n_rois"], "vectors": label,
                     "clusters_on": sorted(on_cl), "clusters_off": sorted(off_cl),
                     "separates": len(on_cl) == 1 and len(off_cl) == 1 and on_cl != off_cl})
cl = pd.DataFrame(rows)
print(cl.to_string(index=False))
for label, g in cl.groupby("vectors"):
    print(f"{label}: ON and OFF separated in {int(g.separates.sum())} of {len(g)} sessions; "
          f"198-ROI session: {bool(g[g.rois == 198].separates.iloc[0])}")

**Result: not reproduced.** In none of the seven sessions, the 198-glomerulus session included,
does the two-cluster cut of the dendrogram put the four ON vectors in one cluster and the four OFF
vectors in the other, with dF/F or with z-scored traces (0 of 7 each).

## fig6e-contrast-msc

> Methyl salicylate presented 1 s after another odor evokes a response less correlated with that
> odor's solitary response than the solitary methyl salicylate response is (history dependent
> contrast enhancement; one-tailed t-test p < 0.01, 3 sessions from 2 mice).

In [ ]:
SEQ_B = slice(128, 208)          # the second odor's 5 s, 6 to 11 s after the pair's onset
rows = []
for s in block:
    seq = olfdyn.dff_pre(s["seq"])
    seq_z = olfdyn.zscore_trials(s["seq"][:, :912])
    lag = (s["pres"][s["pres"].kind == "seq_second"].onset.to_numpy()
           - s["pres"][s["pres"].kind == "seq_first"].onset.to_numpy())
    msc_sol = resp[s["name"], "msc"][ON].mean(0)
    for x, lag_x in zip(s["seq_pres"].odor.unique(), [lag[(s["seq_pres"].odor == o).to_numpy()].mean() for o in s["seq_pres"].odor.unique()]):
        sel = (s["seq_pres"].odor == x).to_numpy()
        x_sol = resp[s["name"], x][ON].mean(0)
        msc_seq = seq[sel].mean(axis=0)[SEQ_B].mean(0)
        x_seq_first = seq[sel].mean(axis=0)[ON].mean(0)
        zx = resp_z[s["name"], x][ON].mean(0)
        zm = resp_z[s["name"], "msc"][ON].mean(0)
        zs = seq_z[sel].mean(axis=0)[SEQ_B].mean(0)
        rows.append({"session": s["name"], "mouse": s["subject"], "first": x, "pairs": int(sel.sum()), "lag_s": lag_x,
                     "r_solitary": olfdyn.corr(x_sol, msc_sol), "r_sequential": olfdyn.corr(x_sol, msc_seq),
                     "r_msc_sol_vs_seq": olfdyn.corr(msc_sol, msc_seq),
                     "r_first_sol_vs_seq": olfdyn.corr(x_sol, x_seq_first),
                     "z_solitary": olfdyn.corr(zx, zm), "z_sequential": olfdyn.corr(zx, zs)})
ce = pd.DataFrame(rows)
print(ce.round(3).to_string(index=False))
d = ce.r_solitary - ce.r_sequential
t = stats.ttest_rel(ce.r_solitary, ce.r_sequential, alternative="greater")
print(f"\nAll 21 session-odor pairs: solitary r {ce.r_solitary.mean():.3f}, sequential r {ce.r_sequential.mean():.3f}, "
      f"difference {d.mean():.3f} (positive in {int((d > 0).sum())} of 21), one-tailed paired t p = {t.pvalue:.2g}")
ps = ce.groupby("session")[["r_solitary", "r_sequential"]].mean()
t2 = stats.ttest_rel(ps.r_solitary, ps.r_sequential, alternative="greater")
print(f"Per session (mean over first odors, n=7): difference {(ps.r_solitary - ps.r_sequential).mean():.3f}, p = {t2.pvalue:.2g}")
for x, g in ce.groupby("first"):
    tx = stats.ttest_rel(g.r_solitary, g.r_sequential, alternative="greater")
    print(f"first odor {x}: difference {(g.r_solitary - g.r_sequential).mean():.3f}, p = {tx.pvalue:.2g}")
tz = stats.ttest_rel(ce.z_solitary, ce.z_sequential, alternative="greater")
print(f"Variant, per-trial z-scored traces: solitary {ce.z_solitary.mean():.3f}, sequential {ce.z_sequential.mean():.3f}, "
      f"one-tailed paired p = {tz.pvalue:.2g}")
pm = ce.groupby("mouse")[["r_solitary", "r_sequential"]].mean()
print("Per mouse (mean):\n", pm.round(3).to_string())

**Result: not reproduced.** Over the 21 session-odor pairs the correlation of solitary X with
solitary methyl salicylate (mean 0.229) does not exceed its correlation with methyl salicylate in
the (X)msc pairs (0.219; difference 0.009, positive in 10 of 21, one-tailed p = 0.39). Per session
p = 0.40, and per first odor p = 0.33 to 0.51. The sequential methyl salicylate response stays
close to the solitary one (r = 0.544 to 0.945). With per-trial z-scored traces the solitary r is
0.289 and the sequential r 0.118 (p = 5.6e-07), as the claim states.

## fig7d-persistent-anticorrelation

> After odor offset the glomerular population activity stays negatively correlated with the
> odor-evoked ON response for several tens of seconds, although overall activity returns to
> baseline quickly.

In [ ]:
BINS = [slice(112 + 80 * i, 192 + 80 * i) for i in range(10)]          # 0-5 s ... 45-50 s after offset
rows = []
for s in block:
    for odor in olfdyn.BLOCK_ODORS:
        r = resp[s["name"], odor]
        med = olfdyn.median_subtracted(r)
        sel = (s["sol_pres"].odor == odor).to_numpy()
        f0 = s["pre_block"][odor][0]
        pb = ((s["sol"][sel] - f0) / f0).mean(axis=0).astype(np.float64)
        pbm = olfdyn.median_subtracted(pb)
        rz = olfdyn.median_subtracted(resp_z[s["name"], odor])
        for i, b in enumerate(BINS):
            rows.append({"session": s["name"], "mouse": s["subject"], "odor": odor, "bin_s": f"{5 * i}-{5 * i + 5}",
                         "r_main": olfdyn.corr(med[ON].mean(0), med[b].mean(0)),
                         "r_no_median": olfdyn.corr(r[ON].mean(0), r[b].mean(0)),
                         "r_preblock_f0": olfdyn.corr(pbm[ON].mean(0), pbm[b].mean(0)),
                         "r_zscored": olfdyn.corr(rz[ON].mean(0), rz[b].mean(0)),
                         "mean_dff": float(r[b].mean()), "mean_dff_on": float(r[ON].mean())})
pc = pd.DataFrame(rows)
order = [f"{5 * i}-{5 * i + 5}" for i in range(10)]
out = []
for b in order:
    g = pc[pc.bin_s == b]
    row = {"bin_s": b}
    for c in ["r_main", "r_no_median", "r_preblock_f0", "r_zscored"]:
        row[c + "_median"] = g[c].median()
        row[c + "_p"] = stats.wilcoxon(g[c]).pvalue
        row[c + "_neg"] = int((g[c] < 0).sum())
    row["mean_dff"] = g.mean_dff.mean()
    out.append(row)
print("Median over 28 session-odor values per bin after offset, two-sided Wilcoxon p, and how many are below 0:")
print(pd.DataFrame(out).round(4).to_string(index=False))
print("\nMean dF/F during ON (over sessions and odors):", round(pc.mean_dff_on.mean(), 4))
print("\nr_main median per mouse and bin:")
print(pc.pivot_table(index="mouse", columns="bin_s", values="r_main", aggfunc="median")[order].round(3).to_string())
print("\nPre-block baseline length (s) per session and odor:",
      {s["name"]: {o: round(v[1]) for o, v in s["pre_block"].items()} for s in block})

**Result: not reproduced.** After offset the population vector stays positively correlated with
the ON vector: the median r over the 28 session-odor pairs is 0.481 at 0 to 5 s, 0.312 to 0.328 from
5 to 20 s, and 0.094 at 45 to 50 s, above 0 in every bin (p <= 0.014). The mean dF/F falls from
0.071 in the first 5 s after offset to 0.005 at 45 to 50 s, so the activity after offset is a
fading positive copy of the ON pattern; in mice 148 and 149 r stays near 0.2 to 0.4 for the whole
50 s. With F0 from the 15 min before each block r is positive in every bin (medians 0.304 to
0.600). With per-trial z-scored traces r is negative from 5 s on (medians -0.344 to -0.480, 26 to
28 of 28 below 0), as the claim states.

## fig7h-iti-knn-mouse

> Glomerular activity averaged over the 50 s after each odor offset classifies the preceding odor
> with high accuracy (k-NN, k = 20).

In [ ]:
ITI = slice(112, 912)            # 0 to 50 s after offset
rng = np.random.default_rng(7)
rows, null_main = [], None
for s in block:
    x = olfdyn.dff_session(s["sol"][:, ITI], s).mean(axis=1).astype(np.float64)       # 40 x ROIs
    y = s["sol_pres"].odor.to_numpy()
    nb = olfdyn.knn_neighbours(x, np.arange(len(x)), 20, "cosine")
    acc = (olfdyn.knn_predict(nb, y) == y).mean()
    null = olfdyn.permuted_accuracy(nb, y, np.arange(len(x)), rng, 1000)
    xc = x - x.mean(0)
    pcs = xc @ np.linalg.svd(xc, full_matrices=False)[2][:3].T
    nb3 = olfdyn.knn_neighbours(pcs, np.arange(len(x)), 20, "cosine")
    acc3 = (olfdyn.knn_predict(nb3, y) == y).mean()
    rows.append({"session": s["name"], "rois": s["n_rois"], "acc_full_k20": acc, "null_p99": np.percentile(null, 99),
                 "null_mean": null.mean(), "acc_top3pc_k20": acc3,
                 "per_odor": {o: round(float((olfdyn.knn_predict(nb, y) == y)[y == o].mean()), 2) for o in olfdyn.BLOCK_ODORS}})
ik = pd.DataFrame(rows)
print(ik.round(3).to_string(index=False))
print("Chance 0.25. Main session: the one with 318 ROIs.")

**Result: partially reproduced.** In the 318-glomerulus session (sub-148 ses-3) the 50-s ITI
averages classify the preceding odor with 45% accuracy (k = 20), above the 99th percentile of the
permutation null (37.5%; null mean 17.1%) but below 50%. Accuracy differs between odors (methyl
salicylate 0.9, acetophenone 0.0, allyl sulfide 0.3, eugenol 0.6). The code's choice, cosine
distance on the top three PCs, gives 12.5%. Across the seven sessions accuracy is 22.5% to 47.5%,
above the null's 99th percentile in three (sub-141 ses-5, sub-148 ses-3, sub-149 ses-1).

## fig7j-iti-timebin-mouse

> A time-bin k-NN classifier identifies the preceding odor from glomerular activity throughout the
> 55 s inter-trial interval; accuracy stays high for its whole duration and falls to chance with
> shuffled labels.

In [ ]:
rng = np.random.default_rng(8)
SEG = 40                          # 4-frame bins per 10 s
rows, curves = [], {}
t0 = time.time()
for s in block:
    x = olfdyn.dff_session(s["sol"][:, ITI], s)                        # 40 x 800 x ROIs
    n_tr, n_fr, n_roi = x.shape
    xb = x.reshape(n_tr, n_fr // 4, 4, n_roi).mean(axis=2)             # 40 x 200 x ROIs
    X = xb.reshape(-1, n_roi)
    trial = np.repeat(np.arange(n_tr), n_fr // 4)
    y = np.repeat(s["sol_pres"].odor.to_numpy(), n_fr // 4)
    seg = np.tile(np.arange(n_fr // 4) // SEG, n_tr)
    nb = olfdyn.knn_neighbours(X, trial, 10, "correlation")
    hit = olfdyn.knn_predict(nb, y) == y
    null = olfdyn.permuted_accuracy(nb, y, trial, rng, 200, by_bin=seg)
    acc_seg = [hit[seg == k].mean() for k in range(5)]
    curves[s["name"]] = np.array([hit[np.tile(np.arange(n_fr // 4), n_tr) == k].mean() for k in range(n_fr // 4)])
    row = {"session": s["name"], "rois": s["n_rois"]}
    for k in range(5):
        row[f"acc_{10 * k}-{10 * k + 10}s"] = acc_seg[k]
        row[f"p99_{10 * k}-{10 * k + 10}s"] = np.percentile(null[:, k], 99)
    row["per_odor_last10s"] = {o: round(float(hit[(seg == 4) & (y == o)].mean()), 2) for o in olfdyn.BLOCK_ODORS}
    rows.append(row)
tb = pd.DataFrame(rows)
print(tb.round(3).to_string(index=False))
print(f"(chance 0.25; {time.time() - t0:.0f} s)")
fig, ax = plt.subplots(figsize=(7, 3))
for n, c in curves.items():
    ax.plot(np.arange(len(c)) * 0.25, 100 * c, lw=1, label=n)
ax.axhline(25, color="grey", ls=":")
ax.set_xlabel("time after offset (s)"); ax.set_ylabel("accuracy (%)"); ax.set_ylim(0, 100)
ax.legend(fontsize=6, ncol=2)
plt.tight_layout()
print("Plotted accuracies range from", round(100 * min(c.min() for c in curves.values()), 1), "to",
      round(100 * max(c.max() for c in curves.values()), 1), "%")

In [ ]:
# Variant: the same classifier on pseudo-trials in the inter-block interval after each solitary block,
# continuing the block's 60.4 s rhythm with no odor given. High accuracy here would show that the
# classifier can name the block from slow differences between blocks, without odor-evoked activity.
rows = []
for s in block:
    counts = {o: len(s["pseudo"][o]) for o in olfdyn.BLOCK_ODORS}
    use = [o for o in olfdyn.BLOCK_ODORS if counts[o] > 0]
    x = np.concatenate([olfdyn.dff_session(s["pseudo"][o], s) for o in use])
    n_tr, n_fr, n_roi = x.shape
    X = x.reshape(n_tr, n_fr // 4, 4, n_roi).mean(axis=2).reshape(-1, n_roi)
    trial = np.repeat(np.arange(n_tr), n_fr // 4)
    y = np.repeat(np.concatenate([[o] * counts[o] for o in use]), n_fr // 4)
    seg = np.tile(np.arange(n_fr // 4) // SEG, n_tr)
    nb = olfdyn.knn_neighbours(X, trial, 10, "correlation")
    hit = olfdyn.knn_predict(nb, y) == y
    row = {"session": s["name"], "pseudo_trials": counts, "chance": round(max(counts.values()) / n_tr, 3) if use else np.nan}
    for k in range(5):
        row[f"acc_{10 * k}-{10 * k + 10}s"] = hit[seg == k].mean()
    rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("chance: the share of the largest class")

**Result: reproduced.** In the 318-glomerulus session the time-bin classifier names the
preceding odor with 88.5% accuracy in the first 10 s after offset and 71.9% to 75.6% from 10 to 50
s, each above 50% and above the null's 99th percentile (32.5% to 35.9%). The other six sessions give
72.2% to 99.4% in every segment. The same classifier on pseudo-trials in the inter-block intervals,
where no odor is given for 15 min, names the preceding block with 69.6% to 96.9% accuracy, as high
as during the ITIs (sub-148 ses-3: 69.6% to 71.2%). The accuracy therefore does not need
odor-evoked activity in the ITI: slow differences between blocks are enough, and these outputs do
not show that it reflects an odor-specific afterimage.

## figs7e-ibi-chance

> During the 15 min between blocks odor-specific information is no longer present; classification
> of 1-min bins of the inter-block interval by the preceding odor is near chance (34.8%).

In [ ]:
rng = np.random.default_rng(12)
print("IBI minutes available per session and preceding odor:",
      {s["name"]: {o: len(v) for o, v in s["ibi"].items()} for s in block})
rows = []
for s in block:
    counts = [len(s["ibi"][o]) for o in olfdyn.BLOCK_ODORS]
    X = np.vstack([s["ibi"][o] for o in olfdyn.BLOCK_ODORS]).astype(np.float64)
    X = (X - s["F_median"]) / s["F_median"]
    y = np.repeat(olfdyn.BLOCK_ODORS, counts)
    complete = all(c == 15 for c in counts)
    for k in ([50, 20] if complete else [20]):
        nb = olfdyn.knn_neighbours(X, np.arange(len(X)), k, "cosine")
        pred = olfdyn.knn_predict(nb, y)
        rate = np.mean([(pred == y)[y == o].mean() for o in olfdyn.BLOCK_ODORS if (y == o).any()])
        null = olfdyn.permuted_accuracy(nb, y, np.arange(len(X)), rng, 1000)
        rows.append({"session": s["name"], "complete": complete, "bins": len(X), "k": k, "rate_pct": 100 * rate,
                     "null_mean_pct": 100 * null.mean(), "null_p95_pct": 100 * np.percentile(null, 95),
                     "perm_p": float((null >= (pred == y).mean()).mean())})
ib = pd.DataFrame(rows)
print(ib.round(2).to_string(index=False))
main_ib = ib[ib.complete & (ib.k == 50)]
print(f"\nMain: mean rate over the {len(main_ib)} sessions with four complete IBIs (k=50): {main_ib.rate_pct.mean():.1f}% "
      f"(range {main_ib.rate_pct.min():.1f}-{main_ib.rate_pct.max():.1f}); expected 34.8, criterion <= 38.3; chance 25")

**Result: blocked (methods underspecified).** With the Methods' k = 50 on 60 one-minute bins the
rate is 0.0% to 13.3% (mean 3.8%), and this procedure's own permutation null averages 10.8%, not the
25% chance of four odors, because the other odors supply most of the 50 neighbours of every bin. The
paper's 34.8%, which it calls roughly chance, cannot come from this procedure, so the paper's
procedure is not the one its Methods describe. With k = 20 the IBI bins are classified by the
preceding block at 73.3% to 95.0% in the four sessions with complete IBIs and 35.0% to 66.7% in the
other three (permutation p <= 0.02), far above chance.

## figs8e-random-order-decline

> When eleven odors are presented in random order (1 s pulses, 19 s apart), odor information after
> offset is variable across odors and on average declines to chance after several seconds.

In [ ]:
rng = np.random.default_rng(13)
AFTER = slice(48, 48 + 272)          # 0 to 17 s after offset (offset 1 s after onset)
rows, curves = [], {}
for s in random_sessions:
    q = s["rnd_pres"]
    x = olfdyn.dff_session(s["rnd"][:, AFTER], s)
    n_tr, n_fr, n_roi = x.shape
    xb = x.reshape(n_tr, n_fr // 4, 4, n_roi).mean(axis=2)
    X = xb.reshape(-1, n_roi)
    trial = np.repeat(np.arange(n_tr), n_fr // 4)
    y = np.repeat(q.odor.to_numpy(), n_fr // 4)
    tbin = np.tile(np.arange(n_fr // 4), n_tr)
    part = np.where(tbin < 8, 0, np.where(tbin >= 48, 2, 1))           # 0-2 s, 2-12 s, 12-17 s
    nb = olfdyn.knn_neighbours(X, trial, 10, "correlation")
    hit = olfdyn.knn_predict(nb, y) == y
    null = olfdyn.permuted_accuracy(nb, y, trial, rng, 200, by_bin=part)
    curves[s["name"]] = np.array([hit[tbin == k].mean() for k in range(n_fr // 4)])
    early, late = hit[part == 0].mean(), hit[part == 2].mean()
    rows.append({"session": s["name"], "rows": len(X), "odors": q.odor.nunique(), "early_0-2s": early,
                 "null_p99_early": np.percentile(null[:, 0], 99), "late_12-17s": late,
                 "null_p95_late": np.percentile(null[:, 2], 95),
                 "early_above": early > np.percentile(null[:, 0], 99), "late_above": late > np.percentile(null[:, 2], 95),
                 "per_odor_early": {o: round(float(hit[(part == 0) & (y == o)].mean()), 2) for o in sorted(q.odor.unique())}})
rd = pd.DataFrame(rows)
print(rd.drop(columns="per_odor_early").round(3).to_string(index=False))
print("\nPer odor, 0-2 s after offset:")
for _, r in rd.iterrows():
    print(" ", r.session, r.per_odor_early)
print(f"\nChance {1 / 11:.3f}. Sessions with early above null: {int(rd.early_above.sum())}; with late not above: {int((~rd.late_above).sum())}; "
      f"with both: {int((rd.early_above & ~rd.late_above).sum())} of {len(rd)}")
fig, ax = plt.subplots(figsize=(7, 3))
for n, c in curves.items():
    ax.plot(np.arange(len(c)) * 0.25, 100 * c, lw=1, label=n)
ax.axhline(100 / 11, color="grey", ls=":")
ax.set_xlabel("time after offset (s)"); ax.set_ylabel("accuracy (%)"); ax.set_ylim(0, 100); ax.legend(fontsize=7)
plt.tight_layout()
print("Plotted accuracies range from", round(100 * min(c.min() for c in curves.values()), 1), "to",
      round(100 * max(c.max() for c in curves.values()), 1), "%")

**Result: partially reproduced.** In all five random-order sessions the odor is classified in the
first 2 s after offset above the null (24.4% to 56.8% against 99th percentiles of 10.6% to 12.3%;
chance 9.1%), with large differences between odors. In the last 5 s accuracy falls to 9.6% to 15.3%,
but it is still above the null's 95th percentile in four of the five sessions; only in sub-134 ses-6
is it not. These sessions are not the paper's (none has its 286 glomeruli), and in three of them
odors have different concentration values.

## Locust claims (data on Figshare)

The locust recordings are not in the dandiset. They are read from the paper's Figshare record,
checked against the md5 it gives. Spike counts are in 50 ms bins; trials are on the second axis.

In [ ]:
loc2 = olfdyn.figshare_mat("data2_locust_PN66.mat")
loc4 = olfdyn.figshare_mat("data4_locust_PN85.mat")
loc5 = olfdyn.figshare_mat("data5_locust_PN85.mat")
odors66 = list(loc2["odors"])
print("PN66 file: odors", odors66, "array", loc2[odors66[0]].shape, "ON bins", loc2["onBins"][[0, -1]], "OFF bins", loc2["offBins"][[0, -1]])
print("PN85 contrast file: single", list(loc4["singleOdors"]), "pairs", [tuple(r) for r in loc4["contrastOrg"]],
      "onInd", loc4["onInd"][[0, -1]], "contrastInd", loc4["contrastInd"][[0, -1]])
print("PN85 persistence file: single", list(loc5["singleOdors"]), "onInd", loc5["onInd"][[0, -1]],
      "array", getattr(loc5["data"], "app").shape)
print("Spike counts per bin range:", int(min(loc2[o].min() for o in odors66)), "to", int(max(loc2[o].max() for o in odors66)))
LON, LOFF = slice(200, 280), slice(280, 360)          # MATLAB 201:280 and 281:360

## fig1g-onoff-angle-locust

> In the locust antennal lobe the median angle between PN population ON and OFF response vectors
> exceeds 90 degrees for every one of 11 odors (mean 106.7 +/- 0.84 SEM).

In [ ]:
rows = []
for o in odors66:
    m = loc2[o].mean(axis=1).T.astype(np.float64)          # bins x PNs, trial-averaged
    base = m[:200].mean(axis=0)
    unc = olfdyn.onoff_angles(m[LON], m[LOFF], center=False)
    zero = int((np.linalg.norm(m[LON], axis=1) == 0).sum() + (np.linalg.norm(m[LOFF], axis=1) == 0).sum())
    rows.append({"odor": o, **olfdyn.onoff_angles(m[LON], m[LOFF], center=True),
                 "uncentred": unc["on_off"], "zero_vectors_uncentred": zero,
                 "baseline_subtracted": olfdyn.onoff_angles(m[LON] - base, m[LOFF] - base, center=False)["on_off"]})
la = pd.DataFrame(rows)
print(la.round(2).to_string(index=False))
print(f"\nMain: mean {la.on_off.mean():.2f}, SEM {la.on_off.std(ddof=1) / np.sqrt(len(la)):.2f}, "
      f"min {la.on_off.min():.2f}, odors above 90: {int((la.on_off > 90).sum())} of {len(la)}; expected 106.7 +/- 0.84")
print(f"Uncentred: mean {la.uncentred.mean():.2f}; baseline-subtracted: mean {la.baseline_subtracted.mean():.2f}, "
      f"above 90: {int((la.baseline_subtracted > 90).sum())}")

**Result: partially reproduced.** All 11 odors give a median ON-OFF angle above 90 degrees (99.2
to 108.7), but the mean, 104.9 degrees (SEM 0.87), is 1.8 degrees below 106.7 +/- 0.84. Without
centring, the mean angle is 58.5 degrees; with each PN's pre-odor mean subtracted and no centring
it is 91.9, with 9 of 11 above 90.

## figs4a-reference-odor-locust

> In the locust the ON response to a reference odor is closer to other odors' ON responses than to
> OFF responses for most reference odors, significantly for 5 of 11.

In [ ]:
x = np.vstack([np.vstack([loc2[o].mean(axis=1).T[LON], loc2[o].mean(axis=1).T[LOFF]]) for o in odors66]).astype(np.float64)
xc = x - x.mean(axis=0)
pcs = xc @ np.linalg.svd(xc, full_matrices=False)[2][:3].T
means = np.stack([pcs[i * 80:(i + 1) * 80].mean(0) for i in range(22)])
on, off = means[0::2], means[1::2]
rows = []
for i, o in enumerate(odors66):
    a_on = olfdyn.angles(on[i:i + 1], on)[0]
    a_off = olfdyn.angles(on[i:i + 1], off)[0]
    t = stats.ttest_ind(a_on, a_off)
    rows.append({"ref": o, "on_mean": a_on.mean(), "off_mean": a_off.mean(), "p": t.pvalue,
                 "sig_on_smaller": (t.pvalue < 0.05) and a_on.mean() < a_off.mean()})
rl = pd.DataFrame(rows)
print(rl.round(4).to_string(index=False))
print(f"\nReference odors with ON angles smaller on average: {int((rl.on_mean < rl.off_mean).sum())} of 11; "
      f"significant (p<0.05, ON smaller): {int(rl.sig_on_smaller.sum())}; expected 5 +/- 1")

**Result: reproduced.** Four reference odors have ON angles significantly smaller than OFF angles
(hexanol, hexanoic acid, phenethylamine, 4-vinyl anisole), within 5 +/- 1, and 9 of 11 have smaller
ON angles on average. Ammonia differs significantly in the other direction (ON 107.3, OFF 52.5
degrees, p = 0.0033).

## fig5e-contrast-locust

> In the locust, PN responses to geraniol presented 0.5 s after another odor are less correlated
> with that odor's solitary response than solitary geraniol is (one-tailed t-test p < 0.05, 5 odor
> pairs).

In [ ]:
d4 = loc4["data"]
on_i, ct_i = loc4["onInd"] - 1, loc4["contrastInd"] - 1
def vec(name, idx):
    return getattr(d4, name).astype(np.float64).mean(axis=1)[:, idx].mean(axis=1)
for second in ["ger", "hex"]:
    rows = []
    for first in ["app", "bza", "cit", "iaa", "oct"]:
        rows.append({"first": first, "r_solitary": olfdyn.corr(vec(first, on_i), vec(second, on_i)),
                     "r_sequential": olfdyn.corr(vec(first, on_i), vec(first + second, ct_i))})
    lc = pd.DataFrame(rows)
    t = stats.ttest_rel(lc.r_solitary, lc.r_sequential, alternative="greater")
    t2 = stats.ttest_rel(lc.r_solitary, lc.r_sequential)
    print(f"--- second odor {second}{' (main)' if second == 'ger' else ' (variant)'}")
    print(lc.round(3).to_string(index=False))
    print(f"solitary {lc.r_solitary.mean():.3f}, sequential {lc.r_sequential.mean():.3f}, one-tailed paired p = {t.pvalue:.3g} "
          f"(two-tailed {t2.pvalue:.3g}, as MATLAB's ttest default in Script_Fig4.m)")

**Result: reproduced.** Solitary first odors correlate with solitary geraniol at r = 0.486 on
average and with geraniol presented after them at 0.227, lower for all five first odors (one-tailed
paired p = 0.0055). With hexanol as the second odor the difference is smaller (0.407 against 0.321,
p = 0.13).

## fig7f-iti-knn-locust

> In the locust, PN activity averaged over the 20 s after odor offset classifies the preceding odor
> (k-NN on angular distance), so ITI activity is odor specific.

In [ ]:
rng = np.random.default_rng(17)
d5 = loc5["data"]
single = list(loc5["singleOdors"])
X = np.vstack([getattr(d5, o).astype(np.float64)[:, :, 380:780].mean(axis=2).T for o in single])     # 70 x 85
y = np.repeat(single, 10)
zero = int((np.linalg.norm(X, axis=1) == 0).sum())
nb = olfdyn.knn_neighbours(X, np.arange(len(X)), 10, "cosine")
acc = (olfdyn.knn_predict(nb, y) == y).mean()
null = olfdyn.permuted_accuracy(nb, y, np.arange(len(X)), rng, 1000)
print(f"Main (k=10, full-dimensional angular distance, 20 s): accuracy {acc:.3f}, null mean {null.mean():.3f}, "
      f"99th percentile {np.percentile(null, 99):.3f}; chance {1 / 7:.3f}; zero vectors {zero}")
print("Per odor:", {o: round(float((olfdyn.knn_predict(nb, y) == y)[y == o].mean()), 2) for o in single})
X21 = np.vstack([getattr(d5, o).astype(np.float64)[:, :, 380:799].mean(axis=2).T for o in single])
xc = X21 - X21.mean(0)
pcs = xc @ np.linalg.svd(xc, full_matrices=False)[2][:3].T
nb3 = olfdyn.knn_neighbours(pcs, np.arange(len(X)), 4, "cosine")
print(f"Variant (code: k=4, cosine on top three PCs, 21 s): accuracy {(olfdyn.knn_predict(nb3, y) == y).mean():.3f}")

**Result: reproduced.** The 20-s ITI averages classify all 70 trials correctly (100%; null 99th
percentile 25.7%, chance 14.3%). The code's choice gives 97.1%.

## fig7i-iti-timebin-locust

> In the locust a time-bin k-NN classifier's accuracy decays across the 20 s after offset to about
> 40% averaged over odors, but for most odors does not fall to chance.

In [ ]:
rng = np.random.default_rng(19)
rows_X, y, trial, tbin = [], [], [], []
for i, o in enumerate(single):
    a = getattr(d5, o).astype(np.float64)[:, :, 380:799]              # PNs x trials x 419 bins
    nb_ = int(np.ceil(a.shape[2] / 4))
    b = np.stack([a[:, :, j * 4:(j + 1) * 4].mean(axis=2) for j in range(nb_)], axis=2)   # last bin partial, as the code
    for tr in range(10):
        rows_X.append(b[:, tr, :].T)
        y += [o] * nb_; trial += [i * 10 + tr] * nb_; tbin += list(range(nb_))
X = np.vstack(rows_X); y = np.array(y); trial = np.array(trial); tbin = np.array(tbin)
const = X.std(axis=1) == 0
print(f"Time bins: {len(X)}; bins with no spikes or constant counts (dropped from queries and neighbours): {int(const.sum())}")
Xv, yv, trv, tbv = X[~const], y[~const], trial[~const], tbin[~const]
nb = olfdyn.knn_neighbours(Xv, trv, 10, "correlation")
hit = olfdyn.knn_predict(nb, yv) == yv
last = tbv >= tbv.max() - 24                                          # last 25 bins, 5 s
per_odor = {o: hit[last & (yv == o)].mean() for o in single}
null_rows = []
trials_u = np.unique(trv)
lab = np.array([yv[trv == t][0] for t in trials_u])
pos = np.searchsorted(trials_u, trv)
for _ in range(200):
    perm = rng.permutation(lab)[pos]
    h = olfdyn.knn_predict(nb, perm) == perm
    null_rows.append([h[last & (perm == o)].mean() for o in single] + [h[last].mean()])
null_rows = np.array(null_rows)
print(f"Mean over odors in the last 5 s: {np.mean(list(per_odor.values())):.3f} (expected 0.40 +/- 0.04); chance {1 / 7:.3f}")
print(f"Over the first 2 s after offset: {hit[tbv < 10].mean():.3f}; over 0-21 s: {hit.mean():.3f}")
print("Per odor in the last 5 s, with the 99th percentile of the null:")
for j, o in enumerate(single):
    print(f"  {o}: {per_odor[o]:.3f} (null p99 {np.percentile(null_rows[:, j], 99):.3f})")
print("Odors above the null:", sum(per_odor[o] > np.percentile(null_rows[:, j], 99) for j, o in enumerate(single)), "of 7")
curve = np.array([hit[tbv == k].mean() for k in range(tbv.max() + 1)])
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(np.arange(len(curve)) * 0.2, 100 * curve, color="k")
ax.axhline(100 / 7, color="grey", ls=":"); ax.set_ylim(0, 100)
ax.set_xlabel("time after offset (s)"); ax.set_ylabel("accuracy (%)")
plt.tight_layout()
print("Plotted accuracies range from", round(100 * curve.min(), 1), "to", round(100 * curve.max(), 1), "%")

**Result: partially reproduced.** Accuracy decays from 78.7% in the first 2 s after offset to
29.0% averaged over odors in the last 5 s, below 40 +/- 4. In the last 5 s every one of the 7 odors
stays above its null's 99th percentile (16.0% to 54.0% against 14.0% to 27.6%), so for most odors
accuracy does not fall to chance.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| mouse-sessions | dandiset | reproduced |
| figshare-trace-link | with inputs | reproduced |
| fig2h-onoff-angle-mouse | dandiset | partially reproduced |
| fig2f-onoff-anticorrelated-mouse | dandiset | not reproduced |
| fig4b-reference-odor-mouse | dandiset | partially reproduced |
| fig4d-clustering-mouse | dandiset | not reproduced |
| fig6e-contrast-msc | dandiset | not reproduced |
| fig6e-contrast-ace | dandiset | blocked: data missing |
| fig7d-persistent-anticorrelation | dandiset | not reproduced |
| fig7h-iti-knn-mouse | dandiset | partially reproduced |
| fig7j-iti-timebin-mouse | dandiset | reproduced |
| figs7e-ibi-chance | dandiset | blocked: methods underspecified |
| figs8e-random-order-decline | dandiset | partially reproduced |
| figs2-pid-decay | dandiset | blocked: data missing |
| fig1g-onoff-angle-locust | elsewhere | partially reproduced |
| figs4a-reference-odor-locust | elsewhere | reproduced |
| fig5e-contrast-locust | elsewhere | reproduced |
| fig7f-iti-knn-locust | elsewhere | reproduced |
| fig7i-iti-timebin-locust | elsewhere | partially reproduced |

**How the analysis differs from the paper's.**

- dF/F is computed from the dandiset's raw F with F0 from the 2 s before each onset (responses)
  or the session median (ITI and IBI). The paper's deposited arrays are close to F z-scored within
  each 57-s trial window. The mouse ON/OFF anticorrelation (fig2f, fig7d) and the contrast
  enhancement (fig6e) appear with that z-scoring and not with dF/F; the z-scored results are shown
  as variants and do not decide the outcomes.
- Windows are placed in seconds; the paper's code counts frames at a nominal 16 per second, which
  drifts by up to about 2 s over a 57-s trial at the sessions' 13.7 to 17.8 Hz.
- Sessions are matched to figures by ROI count. The contrast claim uses all seven block sessions,
  as the paper's three are not identified. The random-order claim uses five sessions that are not
  the paper's.
- The k-NN choices follow the Methods where they can be run; the Methods' k = 50 cannot be run on
  the 40 ITI averages and is degenerate on the 60 IBI bins.

**What was found about the dandiset.**

- The seven block-protocol sessions in four mice match the paper's mouse dataset, and the raw F of
  the 198-glomerulus session matches the paper's deposited array (median within-trial r 0.968, same
  ROI order).
- The fluorescence is raw F (unit "n.a.") with no description of its extraction or of the
  normalisation the paper used, which decides several outcomes here.
- The trials table holds 1-s valve rows without a description; blocks, solitary and sequential
  presentations must be rebuilt from timing. Some sessions fill the gaps with "Empty" rows and some
  do not, and concentrations have no unit and are 0.0 for two odorants in three sessions.
- In three files the raw two-photon movie holds half the frames of the traces.
- No sequential pair ends in acetophenone, and there are no PID recordings, so Figure 6E (right),
  S6 and S2 cannot be tested. The random-order session of Figure S8D/E is not in the dandiset;
  five other random-order sessions are, four of them from mice not in the block sessions, with odor
  names that differ from the paper's for two odors.
- Genotype and age are not stated, and every session starts at 00:00.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")